# 03 — Feature Engineering: Auxiliary Tables

**Proje:** Credit Risk Scoring (Home Credit Default Risk)

Bu defterde geriye kalan dört ilişkisel tabloyu **bağımsız olarak** `SK_ID_CURR` bazına indiriyoruz:

| Tablo | Anlamı |
|---|---|
| `previous_application.csv` | Müşterinin Home Credit'teki **önceki başvuruları** (onay/red, miktar, faiz türü, vade...) |
| `POS_CASH_balance.csv` | Önceki taksitli/POS kredilerinin aylık bakiye geçmişi (DPD ile) |
| `installments_payments.csv` | **Geçmiş ödeme disiplini** — gecikme, eksik tutar, erken ödeme detayları |
| `credit_card_balance.csv` | Önceki kredi kartı aylık kullanım/limit/ödeme geçmişi |

### Tasarım kararları
- Her tablo **kendi parquet'ine** yazılır → ileride sadece bir tabloda değişiklik olduğunda diğerleri yeniden hesaplanmaz.
- Her hücrenin sonunda ham DataFrame `del` ile bellekten serbest bırakılır → 13M+ satırlı tablolarla bile RAM patlamaz.
- Tüm ağır iş `src/feature_engineering.py`'deki `aggregate_*` fonksiyonlarına taşındı; defter sadece orchestration yapar.
- Final birleştirme (application_train + tüm bu özetler) **sıradaki defterde** (`04_Final_Dataset.ipynb`) yapılacak.

> **Memory uyarısı:** `installments_payments` ~13.6M, `POS_CASH_balance` ~10M satır. Çalıştırmadan önce diğer notebook kernel'lerini kapatmanı öneririm.

In [ ]:
import gc
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_csv, PROCESSED_DIR
from src.feature_engineering import (
    aggregate_previous_application,
    aggregate_pos_cash,
    aggregate_installments_payments,
    aggregate_credit_card_balance,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Processed dir: {PROCESSED_DIR}")

In [ ]:
def aggregate_and_save(
    raw_filename: str,
    agg_fn,
    out_filename: str,
):
    """Load raw CSV → aggregate → write parquet → free memory.

    Tek hücrede orchestration yapmak için yardımcı; her tablo aynı şablonla
    işlenir, ama fonksiyon mantığı `feature_engineering.py`'da kalır.
    """
    df = load_csv(raw_filename)
    print(f"  raw shape : {df.shape}")
    agg = agg_fn(df)
    print(f"  agg shape : {agg.shape}")

    out_path = PROCESSED_DIR / out_filename
    agg.to_parquet(out_path, index=False)
    size_mb = out_path.stat().st_size / 1024**2
    print(f"  saved     : {out_path.name}  ({size_mb:.2f} MB)")

    del df, agg
    gc.collect()
    return out_path

## 1) `previous_application` → `PREV_*` özellikleri

Türetilmiş feature: `APP_CREDIT_PERC = AMT_APPLICATION / AMT_CREDIT` (talep edilen / verilen oranı). Ek olarak `NAME_CONTRACT_STATUS = Approved / Refused` filtreli özetler `PREV_APPROVED_*` ve `PREV_REFUSED_*` öneklerinde gelir.

In [ ]:
print("[previous_application]")
prev_path = aggregate_and_save(
    "previous_application.csv",
    aggregate_previous_application,
    "previous_application_agg.parquet",
)

## 2) `POS_CASH_balance` → `POS_*` özellikleri

`SK_DPD` ve `SK_DPD_DEF` (default tanımıyla gecikme) en bilgi verici sütunlar.

In [ ]:
print("[POS_CASH_balance]")
pos_path = aggregate_and_save(
    "POS_CASH_balance.csv",
    aggregate_pos_cash,
    "pos_cash_agg.parquet",
)

## 3) `installments_payments` → `INSTAL_*` özellikleri

Klasik literatürün **en güçlü tablosu** — modelimizin temerrüdü öğrenmesinde anahtar rol oynayacak.

Türetilmiş ödeme disiplin metrikleri:
- **`PAYMENT_PERC`** = `AMT_PAYMENT / AMT_INSTALMENT` (ödeme oranı; <1 ise eksik ödeme)
- **`PAYMENT_DIFF`** = `AMT_INSTALMENT − AMT_PAYMENT` (eksik tutar)
- **`DPD`** = max(`DAYS_ENTRY_PAYMENT − DAYS_INSTALMENT`, 0) — geç ödeme gün sayısı
- **`DBD`** = max(`DAYS_INSTALMENT − DAYS_ENTRY_PAYMENT`, 0) — erken ödeme gün sayısı

> ~13.6M satır. Birkaç saniye sürer.

In [ ]:
print("[installments_payments]")
ins_path = aggregate_and_save(
    "installments_payments.csv",
    aggregate_installments_payments,
    "installments_payments_agg.parquet",
)

## 4) `credit_card_balance` → `CC_*` özellikleri

Tüm numerik sütunlar üzerinde geniş özet (min/max/mean/sum/var). Kart limitinin kullanımı (`AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL`) gibi türetilmiş bir oran ileride final birleştirme aşamasında eklenecek.

In [ ]:
print("[credit_card_balance]")
cc_path = aggregate_and_save(
    "credit_card_balance.csv",
    aggregate_credit_card_balance,
    "credit_card_balance_agg.parquet",
)

## 5) Üretilen feature parquet'lerinin özeti

Bir sonraki defter (`04_Final_Dataset.ipynb`) bu dört dosyayı + `application_train_bureau.parquet`'i okuyup `application_train` ile birleştirecek.

In [ ]:
summary_rows = []
for path in [prev_path, pos_path, ins_path, cc_path]:
    df = pd.read_parquet(path)
    summary_rows.append(
        {
            "file": path.name,
            "rows": len(df),
            "columns": df.shape[1],
            "size_mb": round(path.stat().st_size / 1024**2, 2),
        }
    )
    del df
    gc.collect()

pd.DataFrame(summary_rows).set_index("file")